# Activity: Solving the Bouquet Design Problem
In this activity, we will explore an interesting problem in combinatorial optimization known as the Bouquet Design Problem. The goal is to create the most aesthetically pleasing bouquet using a limited number of flowers while adhering to specific design constraints.

> __Learning Objectives:__
>
> Three key objectives go here

This is going to be fun, so let's get started!
___

## Problem
Fill me in
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Implementation
First, let's implement the `customer(...)` function. This function takes the action $\mathbf{a}$ from our agent (encodes what to include in the bouquet) and returns the reward (and some othetr data) associated with choosing this action, i.e., how do you feel about this bouquet you created? 

In [2]:
function customer(t::Int, a::Vector{Int64}, 
    context::MyConsumerChoiceBanditContextModel)

    # Fill me in later

end

customer (generic function with 1 method)

### Constants
Finally, let's set some constants we'll use in the subsequent tasks. See the comment beside the value for a description of what it is, its permissible values, etc.

In [3]:
K = 7; # number of arms for the bandit (number of flower types and other design elements)
T = 5000; # number of rounds for each decision task

___

## Task 1: Let's build the problem context model
In this task, we'll build the problem context model. This model will help us understand the constraints and objectives of the bouquet design problem.

We encode the problem context model as an instance of [the `MyConsumerChoiceBanditContextModel.jl` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyConsumerChoiceBanditContextModel) which we construct [using a custom `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.build). 

Let's save this model in the `my_context_model::MyConsumerChoiceBanditContextModel` variable.

In [5]:
my_context_model = let

    # initialize -
    number_of_components = K; # how many flower types / design elements do we have?
    N = 2^K; # number of possible bouquets (arms)
    ϵ = 0.000001; # small number to avoid division by zero
    
    # setup problem components -
    γ = rand(Uniform(-1.0, 1.0), number_of_components); # Preferences: γᵢ ∈ [-1.0,1.0] for i=1,...,K. We'll make this random for now (you can adjust later)
    p = rand(Uniform(1.0, 10.0), number_of_components); # Prices: pᵢ ∈ [1.0,10.0] for i=1,...,K. We'll make this random for now (you can adjust later)
    μₒ = zeros(N); # initial mean utilities for each arm (bouquet)   
    B = 100.0; # Budget for the bouquet design (you can adjust later)
    nₒ = ones(K); # initial counts for each item

    # compute the bounds -
    bounds = zeros(K,2); # how much of an element can we have in a bouquet?
    for i in 1:K
        bounds[i,1] = ϵ; # lower bound
        bounds[i,2] = floor(B / p[i]); # upper bound (either 1 or budget limited)
    end

    # let's build our items descriptor list 
    items = Array{String,1}(undef, K);
    for i in 1:K
        items[i] = "Flower-$(i)";
    end

    # next, we'll build our context data model (for now, just prices key'd by item descriptor)
    context_data = Dict{String,Any}();
    for i in 1:K
        context_data[items[i]] = p[i]; # price for each item
    end

    # let's build by context model -
    my_context_model = build(MyConsumerChoiceBanditContextModel, (
        B = B, # budget we can spend on bouquet
        items = items, # item descriptors
        bounds = bounds, # bounds on each item
        γ = γ, # customer preferences
        μₒ = μₒ, # initial mean utilities for each arm
        nₒ = nₒ, # initial counts for each item
        data = context_data # context data dictionary
    ));

    my_context_model; # return the context model
end;

Let's build a table that shows the (random) preferences and prices for each possible flower type in our bouquet design problem. This table will help us understand the different flower options available for our bouquet.

We'll use [the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl) to create a nicely formatted table.

In [6]:
let

    # initialize -
    contextmodel = my_context_model;
    K = length(my_context_model.items); # number of possible design elements
    N = 2^K; # number of possible bouquets (arms)
    df = DataFrame();

    # get some data from out context model -
    items = contextmodel.items;
    bounds = contextmodel.bounds;
    prices = [contextmodel.data[items[i]] for i ∈ 1:K];
    preferences = contextmodel.γ;

    for i ∈ 1:K
       
        # package each row of the table
        row_df = (
            flower = items[i],
            price = prices[i],
            preference = preferences[i],
            min_in_bouquet = bounds[i,1],
            max_in_bouquet = bounds[i,2]
        );
        push!(df, row_df);
    end

    # display the table -
    pretty_table(
         df;
         backend = :text,
         table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

 ---------- --------- ------------ ---------------- ----------------
    flower     price   preference   min_in_bouquet   max_in_bouquet 
    String   Float64      Float64          Float64          Float64 
 ---------- --------- ------------ ---------------- ----------------
  Flower-1    5.1145     0.592124           1.0e-6             19.0
  Flower-2   3.66422    -0.217969           1.0e-6             27.0
  Flower-3   5.76598     0.415557           1.0e-6             17.0
  Flower-4   4.50782    -0.709461           1.0e-6             22.0
  Flower-5   5.05832    -0.675909           1.0e-6             19.0
  Flower-6   1.21401    -0.725414           1.0e-6             82.0
  Flower-7   4.47706    0.0589992           1.0e-6             22.0
 ---------- --------- ------------ ---------------- ----------------


## Task 2: Let's design the optimal bouquet using a bandit agent
Fill me in

## Summary
One direct summary sentence goes here.

> __Key Takeaways:__
>
> Three takeaways go here

One direct summary sentence goes here.
___